In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import joblib

import sys
import os
sys.path.append(os.path.abspath('..'))

from BettingStrategy.ModelStrategy.LogisticRegression.get_train_test import TrainTestBuilder
from betting_strategy import betting_pipeline, seperate_bets_dfs
import shap

c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path
from DataPipeline.FeatureEngineering.features_pipeline import FeatureEngineering

BASE_DIR = Path(r'c:/Users/jcmar/my_files/SportsBetting').resolve()
from DataPipeline.FeatureEngineering.features_pipeline import FeatureEngineering
features = FeatureEngineering()


from DataPipeline.utils.email_utils import email_bets

recent_date = r'2026-02-26'
stats_history_file_string = BASE_DIR / "Data" / "scraped_data_main" / f"stats_history_{recent_date}.csv"
odds_history_file_string = BASE_DIR / "Data" / "scraped_data_main" / f"odds_history_{recent_date}.csv"
missing_stats_history_fp = BASE_DIR / "Data" / "non_merged_features" / f"non_merged_stats.csv"
missing_odds_history_fp = BASE_DIR / "Data" / "non_merged_features" / f"non_merged_odds.csv"
upcoming_scraped_stats_string = BASE_DIR / "Data" / "upcoming_events" / "scraped_data" / "upcoming_stats"
upcoming_scraped_odds_string = BASE_DIR / "Data" /"upcoming_events" /"scraped_data" / "upcoming_odds"

non_merged_stats = pd.read_csv(missing_stats_history_fp)
non_merged_stats = non_merged_stats.drop(columns=[col for col in non_merged_stats.columns if "Unnamed" in col])

non_merged_odds = pd.read_csv(missing_odds_history_fp)
non_merged_odds = non_merged_odds.drop(columns=[col for col in non_merged_odds.columns if "Unnamed" in col])


stats_history = pd.read_csv(stats_history_file_string) # frames BEFORE any feature engineering 
stats_history = stats_history.drop(columns=[col for col in stats_history.columns if "Unnamed" in col])
stats_history = pd.concat([stats_history, non_merged_stats], axis=0, ignore_index=True)
stats_history = stats_history[~stats_history.duplicated(
    subset=['fighter_red', 'fighter_blue', 'event_date'],
    keep='first'
    )]

odds_history = pd.read_csv(odds_history_file_string)
odds_history = odds_history.drop(columns=[col for col in odds_history.columns if "Unnamed" in col])
odds_history = pd.concat([odds_history, non_merged_odds], axis=0, ignore_index=True)


next_stats_df = pd.read_csv(f'{upcoming_scraped_stats_string}.csv')
next_odds_df = pd.read_csv(f'{upcoming_scraped_odds_string}.csv')

In [20]:

upcoming_scraped_stats_string = BASE_DIR / "Data" / "upcoming_events" / "scraped_data" / "upcoming_stats"
upcoming_scraped_odds_string = BASE_DIR / "Data" /"upcoming_events" /"scraped_data" / "upcoming_odds"


odds_stats_df, upcoming_df = features.build_all_stats(stats_history, next_stats_df, odds_history, next_odds_df)


C:\Users\jcmar\my_files\SportsBetting
Fight time feats shape" (8165, 87)
Upcoming single event features shape: (78, 16)


c:\Users\jcmar\my_files\SportsBetting\DataPipeline\FeatureEngineering\features_pipeline.py:71: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([empty_df, past_event_stats], axis=0).reset_index(drop=True) # Combine with past event stats


Rolling features shape: (8243, 104)


c:\Users\jcmar\my_files\SportsBetting\DataPipeline\FeatureEngineering\RatingAlgos\glicko2.py:40: RuntimeWarning: divide by zero encountered in scalar divide
  return 1.0 / s
c:\Users\jcmar\my_files\SportsBetting\DataPipeline\FeatureEngineering\RatingAlgos\glicko2.py:49: RuntimeWarning: invalid value encountered in scalar multiply
  return v * s
c:\Users\jcmar\my_files\SportsBetting\DataPipeline\FeatureEngineering\RatingAlgos\glicko2.py:54: RuntimeWarning: invalid value encountered in scalar subtract
  num = ex * (delta**2 - phi**2 - v - ex)


Total features shape: (8243, 222)
Merged df shape: (8243, 291)
Shapes of fav counts:
(8243,) (8243,) (8243,) (8243,)


In [22]:
df_next = upcoming_df[upcoming_df['date'] == '2026-04-11'].copy().reset_index(drop=True)
df_next.isna().sum()


td_defense_pct_red                 0
td_total_attempted_against_red     0
td_total_landed_against_red        0
td_defense_pct_blue                0
td_total_attempted_against_blue    0
                                  ..
dog_counts_red                     0
fav_counts_blue                    0
dog_counts_blue                    0
fav_counts_diff                    0
dog_counts_diff                    0
Length: 297, dtype: int64

In [3]:
df = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-04-11.csv')

df['win_streak_diff']

0     5.0
1     4.0
2    -1.0
3     1.0
4    -7.0
5    -1.0
6    -1.0
7     0.0
8    -1.0
9     1.0
10    1.0
Name: win_streak_diff, dtype: float64

In [2]:
date = '2026-04-11'


model_open = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_open.pkl")
model_close1 = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_close1.pkl")
model_close2 = sm.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\logit_model_close2.pkl")

scaler_open = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_open.pkl")
scaler_close1 = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_close1.pkl")
scaler_close2 = joblib.load("C:\\Users\\jcmar\\my_files\\SportsBetting\\data\\saved_models\\scaler_close2.pkl")

# feats must be in the same order as passed in originally, 

open_feats = [
                  'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue'
                  ]

close1_feats = [
                  'proba_fair_close1_diff', 'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue',
                  ]

close2_feats = [
                  'proba_fair_close2_diff', 'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue',
                  ]


feats_list = [open_feats, close1_feats, close2_feats]
model_list = [model_open, model_close1, model_close2]
scaler_list = [scaler_open, scaler_close1, scaler_close2]

type_list = ['open', 'close1', 'close2']
fair_odds_list = [['dec_fair_open_blue', 'dec_fair_open_red'], ['dec_fair_close1_blue', 'dec_fair_close1_red'], ['dec_fair_close2_blue', 'dec_fair_close2_red']]
real_odds_list = [['dec_open_blue', 'dec_open_red'], ['dec_close1_blue', 'dec_close1_red'], ['dec_close2_blue', 'dec_close2_red'] ]




In [ ]:
from betting_strategy import betting_pipeline, seperate_bets_dfs

BASE_DIR = Path(r'c:/Users/jcmar/my_files/SportsBetting').resolve()

from DataPipeline.utils.email_utils import email_bets



In [10]:
upcoming_events_folder =  BASE_DIR / "Data" / "upcoming_events" / "event_features" 

upcoming_df = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\upcoming_df_test.csv')
for date, group in upcoming_df.groupby('date'):
    
    group = group.reset_index(drop=True)

    # date_str = date.strftime("%Y-%m-%d")   # or "%Y%m%d"
    date_str = date
    event_file_path = upcoming_events_folder / f"upcoming_odds_stats_{date_str}.csv"
    existing_df = pd.read_csv(event_file_path)

    aligned = group.merge(
        existing_df[["fighter_red", "fighter_blue", "open_red", "open_blue"]],
        on=["fighter_red", "fighter_blue"],
        how="left"
    )

    mask_na_update = (
        (aligned['open_red_y'].isna() & aligned['open_red_x'].notna()) |
        (aligned['open_blue_y'].isna() & aligned['open_blue_x'].notna())
    )
    
    sub_group = group[mask_na_update]

    sub_group['math_red'] = sub_group['math_red'].astype('category')
    sub_group['math_blue'] = sub_group['math_blue'].astype('category')
    sub_group['elo_pred'] = sub_group['elo_pred'].astype('category')

    # df_bets_all, df_parlay_all = betting_pipeline(sub_group, 
    #                                     feats_list=feats_list, 
    #                                     model_list=model_list, 
    #                                     scaler_list=scaler_list, 
    #                                     type_list=type_list,
    #                                     fair_odds_list=fair_odds_list, 
    #                                     real_odds_list=real_odds_list, 
    #                                     bankroll=500, max_drawdown=0.3, N=250)

    # print(date)
    email_bets(sub_group, date)



C:\Users\jcmar\AppData\Local\Temp\ipykernel_63572\397111295.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_group['math_red'] = sub_group['math_red'].astype('category')
C:\Users\jcmar\AppData\Local\Temp\ipykernel_63572\397111295.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_group['math_blue'] = sub_group['math_blue'].astype('category')
C:\Users\jcmar\AppData\Local\Temp\ipykernel_63572\397111295.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFr

Constant-like columns: ['const', 'win_streak_diff', 'elo_pred']
0    0.161674
1   -0.169262
dtype: float64
0    0.678397
1    0.581043
dtype: float64
0    1.712380
1    1.429736
dtype: float64
[0.11675379 0.        ]
0     chris padilla
1    charles radtke
dtype: object
['marquel mederos' 'francisco prado']
['chris padilla' 'charles radtke']
0    2026-04-11
Name: date, dtype: object
Constant-like columns: ['const', 'win_streak_diff', 'elo_pred']
0    0.109188
1   -0.109417
dtype: float64
0    0.690331
1    0.573316
dtype: float64
0    1.606748
1    1.553390
dtype: float64
[0.08647109 0.        ]
0     chris padilla
1    charles radtke
dtype: object
['marquel mederos' 'francisco prado']
['chris padilla' 'charles radtke']
0    2026-04-11
Name: date, dtype: object
Constant-like columns: ['const', 'win_streak_diff', 'elo_pred']
0    0.111342
1   -0.102722
dtype: float64
0    0.692372
1    0.567746
dtype: float64
0    1.605122
1    1.580422
dtype: float64
[0.12417241 0.        ]
0     chris

SMTPAuthenticationError: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 00721157ae682-7a36ea2872esm93799637b3.19 - gsmtp')

In [35]:
sub_group[['fighter_red', 'fighter_blue', 'open_red', 'open_blue', 'math_red', 'math_blue', 'elo_pred']]

,fighter_red,fighter_blue,open_red,open_blue,math_red,math_blue,elo_pred
8,chris padilla,marquel mederos,-150.0,130.0,0,0,1
10,charles radtke,francisco prado,-250.0,210.0,0,0,1


In [6]:
folder_path = r"C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features"

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        print(file_path)

        upcoming_fp = fr'{file_path}'
        upcoming_df = pd.read_csv(upcoming_fp)  

        upcoming_df['math_red'] = upcoming_df['math_red'].astype('category')
        upcoming_df['math_blue'] = upcoming_df['math_blue'].astype('category')
        upcoming_df['elo_pred'] = upcoming_df['elo_pred'].astype('category')

        df_bets_all, df_parlay_all = betting_pipeline(upcoming_df, 
                                            feats_list=feats_list, model_list=model_list, scaler_list=scaler_list, type_list=type_list,
                                            fair_odds_list=fair_odds_list, real_odds_list=real_odds_list, 
                                            bankroll=500, max_drawdown=0.25, N=2000)

        df_bets_all[['open_red', 'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue']] = upcoming_df[['open_red', 'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue']]
        df_bets_arr, df_parlay_arr = seperate_bets_dfs(df_bets_all, df_parlay_all, type_list)

        text = file_path.split("_")
        date = text[-1]



C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-03-14.csv
Constant-like columns: ['const']
Constant-like columns: ['const']
Constant-like columns: ['const']
C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-03-21.csv
Constant-like columns: ['const']
Constant-like columns: ['const']
Constant-like columns: ['const']
C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-03-28.csv
Constant-like columns: ['const']
Constant-like columns: ['const']
Constant-like columns: ['const']
C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-04-04.csv
Constant-like columns: ['const']
Constant-like columns: ['const']
Constant-like columns: ['const']
C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\event_features\upcoming_odds_stats_2026-04-11.csv
Constant-like columns: ['const']
Constant-like columns: ['c

In [18]:
df_parlay_1 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\parlays\parlay_all_2026-03-14.csv')

In [19]:
for c in df_parlay_1.columns:
    print(c)

Unnamed: 0
choice_fighter_name_open
parlay_fstar_open
parlay_odds_open
stake_open
parlay_ev_open
parlay_prob_open
choice_fighter_name_close1
parlay_fstar_close1
parlay_odds_close1
stake_close1
parlay_ev_close1
parlay_prob_close1
choice_fighter_name_close2
parlay_fstar_close2
parlay_odds_close2
stake_close2
parlay_ev_close2
parlay_prob_close2


In [16]:
df_all_1 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\upcoming_events\straight_bets\ml_all_2026-03-14.csv')

In [17]:
for c in df_all_1.columns:
    print(c)

Unnamed: 0
elo_red
proba_fair_close1_diff
ratio_control_diff
reach_diff
lose_streak_diff
elo_blue
td_landed_pm_diff
adjusted_sig_str_red
win_pct_blue
age_red
win_pct_red
adjusted_sig_str_blue
adjusted_td_red
sig_str_accuracy_pct_diff
sig_str_absorbed_total_diff
sig_str_defense_pct_diff
ratio_td_diff
win_streak_diff
sub_att_pm_red
adjusted_td_blue
proba_fair_close2_diff
proba_fair_open_diff
sub_att_pm_blue
elo_pred
age_blue
fighter_red
fighter_blue
date
close1_red
close2_red
close1_blue
close2_blue
pred_name_open
pred_winner_open
choice_proba_open
fstar_open
stake_open
edge_open
ev_open
pred_name_close1
pred_winner_close1
choice_proba_close1
fstar_close1
stake_close1
edge_close1
ev_close1
pred_name_close2
pred_winner_close2
choice_proba_close2
fstar_close2
stake_close2
edge_close2
ev_close2
open_red
open_blue


In [27]:
df_bets_arr[1].head(20)

,fighter_red,fighter_blue,pred_name_close1,pred_winner_close1,choice_proba_close1,close1_red,close1_blue,fstar_close1,stake_close1,ev_close1,edge_close1
0,natalia silva,rose namajunas,natalia silva,1,0.807789,-500.0,310.0,0.000000,0.000000,-0.000232,-0.025545
1,kayla harrison,amanda nunes,kayla harrison,1,0.709203,-245.0,130.0,0.000000,0.000000,0.091481,-0.000942
2,sean omalley,song yadong,sean omalley,1,0.661578,-225.0,163.0,0.000000,0.000000,-0.000636,-0.030730
3,waldo cortes acosta,derrick lewis,waldo cortes acosta,1,0.798426,-333.0,225.0,0.075261,37.630292,0.078303,0.029373
4,ateba gautier,andrey pulyaev,ateba gautier,1,0.832048,-1200.0,500.0,0.000000,0.000000,-0.077144,-0.091029
5,arnold allen,jean silva,jean silva,0,0.645086,163.0,-280.0,0.000000,0.000000,-0.065748,-0.091757
6,umar nurmagomedov,deiveson figueiredo,umar nurmagomedov,1,0.920489,-2500.0,700.0,0.000000,0.000000,-0.027304,-0.041049
7,michael johnson,alexander hernandez,alexander hernandez,0,0.689310,145.0,-215.0,0.021327,10.663345,0.070454,0.006771
8,nikita krylov,modestas bukauskas,modestas bukauskas,0,0.571107,130.0,-195.0,0.000000,0.000000,-0.077394,-0.089910
9,alex perez,charles johnson,charles johnson,0,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN


In [122]:
df_bets_arr[2].head(20)

,fighter_red,fighter_blue,pred_name_close2,pred_winner_close2,choice_proba_close2,close2_red,close2_blue,fstar_close2,stake_close2,ev_close2,edge_close2
0,natalia silva,rose namajunas,natalia silva,1,0.787143,-430.0,350.0,0.000000,0.000000,-0.015695,-0.024178
1,kayla harrison,amanda nunes,kayla harrison,1,0.674967,-155.0,190.0,0.051849,25.924444,0.073083,0.067124
2,sean omalley,song yadong,sean omalley,1,0.670930,-200.0,175.0,0.012789,6.394392,0.026321,0.004263
3,waldo cortes acosta,derrick lewis,waldo cortes acosta,1,0.777148,-300.0,270.0,0.070952,35.475969,0.047044,0.027148
4,ateba gautier,andrey pulyaev,ateba gautier,1,0.830059,-800.0,600.0,0.000000,0.000000,-0.056423,-0.058830
5,arnold allen,jean silva,jean silva,0,0.672887,235.0,-225.0,0.000000,0.000000,-0.033337,-0.019420
6,umar nurmagomedov,deiveson figueiredo,umar nurmagomedov,1,0.915955,-1408.0,950.0,0.000000,0.000000,-0.011502,-0.017732
7,michael johnson,alexander hernandez,alexander hernandez,0,0.683407,165.0,-170.0,0.053980,26.989824,0.090770,0.053777
8,nikita krylov,modestas bukauskas,modestas bukauskas,0,0.534449,155.0,-150.0,0.000000,0.000000,-0.114512,-0.065551
9,alex perez,charles johnson,charles johnson,0,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN
